# Enhanced Federated Learning Pipeline (Clean Driver)

This notebook now focuses only on the essentials to start and monitor FL training.
Core modules have been moved to standalone Python files in this workspace.

## 1) Environment and Paths
Set the key paths and conservative dataset caps before running the pipeline.

In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import tensorflow as tf

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

WORKDIR = Path.cwd()
FRAMES_DIR = Path("/kaggle/working/ffpp_frames")
MODEL_PATH = Path("efficientnetb4_final.keras")

# Keep validation/test intentionally small to reduce RAM pressure.
MAX_VAL_SAMPLES = 512
MAX_TEST_SAMPLES = 256

print(f"Workspace: {WORKDIR}")
print(f"Frames dir exists: {FRAMES_DIR.exists()}")
print(f"Model exists: {MODEL_PATH.exists()}")

## 2) Verify Required Modules
These files were extracted from the old notebook and are now imported directly.

In [ ]:
required_modules = [
    "enhanced_client_selection.py",
    "update_validation.py",
    "knowledge_distillation.py",
    "client_reputation_ledger.py",
    "evaluation_metrics.py",
    "federated_learning_cycle.py",
]

missing = [m for m in required_modules if not Path(m).exists()]
if missing:
    raise FileNotFoundError(f"Missing module files: {missing}")

print("All required module files are present.")

## 3) Build Capped Datasets
Creates lightweight val/test/proxy/sup datasets from frame paths using generator-backed tf.data pipelines.

In [ ]:
import glob

all_paths = sorted(glob.glob(str(FRAMES_DIR / "**/*.jpg"), recursive=True))
assert len(all_paths) > 0, f"No frames found in {FRAMES_DIR}"

rng = np.random.RandomState(SEED)
idx = rng.permutation(len(all_paths))
all_paths = [all_paths[i] for i in idx]

def _path_to_label(path: str) -> np.float32:
    return np.float32(1.0 if "fake" in path.lower() else 0.0)

all_labels = [_path_to_label(p) for p in all_paths]
n = len(all_paths)

n_val = min(max(1, int(n * 0.15)), MAX_VAL_SAMPLES)
n_test = min(max(1, int(n * 0.10)), MAX_TEST_SAMPLES)
n_proxy = max(1, int(n * 0.015))
n_sup = max(1, int(n * 0.02))

val_paths = all_paths[:n_val]
val_labels = all_labels[:n_val]
test_paths = all_paths[n_val:n_val + n_test]
test_labels = all_labels[n_val:n_val + n_test]
proxy_paths = all_paths[n_val + n_test:n_val + n_test + n_proxy]
sup_paths = all_paths[n_val + n_test + n_proxy:n_val + n_test + n_proxy + n_sup]
sup_labels = all_labels[n_val + n_test + n_proxy:n_val + n_test + n_proxy + n_sup]

print(f"Datasets built: val={len(val_paths)}, test={len(test_paths)}, proxy={len(proxy_paths)}, sup={len(sup_paths)}")

MODEL_IMG_SIZE = (224, 224)
if MODEL_PATH.exists():
    try:
        _tmp_model = tf.keras.models.load_model(MODEL_PATH, compile=False)
        MODEL_IMG_SIZE = tuple(_tmp_model.input_shape[1:3])
        del _tmp_model
    except Exception as e:
        print(f"Model load warning (using default 224x224): {e}")

def _load_image(path, label):
    img = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
    img = tf.cast(tf.image.resize(img, MODEL_IMG_SIZE), tf.float32)
    return img, label

def _load_image_only(path):
    img = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
    img = tf.cast(tf.image.resize(img, MODEL_IMG_SIZE), tf.float32)
    return img

def _ds_from_paths_labels(paths, labels):
    return tf.data.Dataset.from_generator(
        lambda: ((p, np.float32(y)) for p, y in zip(paths, labels)),
        output_signature=(
            tf.TensorSpec(shape=(), dtype=tf.string),
            tf.TensorSpec(shape=(), dtype=tf.float32),
        ),
    )

def _ds_from_paths(paths):
    return tf.data.Dataset.from_generator(
        lambda: (p for p in paths),
        output_signature=tf.TensorSpec(shape=(), dtype=tf.string),
    )

val_ds = _ds_from_paths_labels(val_paths, val_labels).map(_load_image, num_parallel_calls=2)
test_ds = _ds_from_paths_labels(test_paths, test_labels).map(_load_image, num_parallel_calls=2)
proxy_ds = _ds_from_paths(proxy_paths).map(_load_image_only, num_parallel_calls=2)
sup_ds = _ds_from_paths_labels(sup_paths, sup_labels).map(_load_image, num_parallel_calls=2)

## 4) Configure and Start FL Pipeline
Imports only the orchestrator and config objects needed to run training.

In [ ]:
from federated_learning_cycle import (
    FLCycleConfig,
    FederatedLearningCycle,
)
from knowledge_distillation import DistillationConfig
from enhanced_client_selection import SelectionWeights
from update_validation import ContributionWeights, ClippingConfig
from client_reputation_ledger import ReputationConfig

config = FLCycleConfig(
    model_path=str(MODEL_PATH),
    reports_dir="reports",
)

cycle = FederatedLearningCycle(config)
_ = cycle.load_global_model()
print("Cycle initialized.")

In [ ]:
# Start training when ready.
history = cycle.run(
    server_val_data=val_ds,
    test_data=test_ds,
    proxy_data=proxy_ds,
    supervised_data=sup_ds,
)
print("Training complete. History keys:", list(history.keys()))